<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/agentic_ai_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic AI and RAGs ?

Using only open source frameworks.
- Tiny KB with FAISS + FakeEmbeddings
- Free Wikipedia utility (no API keys)
- Rule-based planner
- Stub summarizer with optional tiny HF model (sshleifer/tiny-gpt2)
Run all cells top to bottom in Colab.

In [ ]:
!pip install -q langchain langchain-community faiss-cpu wikipedia transformers accelerate sentencepiece

## 1) Build the KB retriever

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FakeEmbeddings

kb_docs = [
    Document(
        page_content="""
Agentic systems reason about which tools to call instead of invoking tools blindly.
Their core loop is to understand the goal, inspect the available context, choose
the minimum useful tools, observe the results, and then produce a grounded answer.
They should reconsider the plan after each tool call and handle failures gracefully.
""".strip(),
        metadata={"source": "kb:agentic_concept"},
    ),
    Document(
        page_content="""
Retrievers fetch grounding passages from a knowledge base. A good retrieval flow
normalizes the question, retrieves a small number of candidates, checks whether
they are relevant, and answers only from supported evidence. If the passages do
not answer the question, the assistant should say so rather than guess.
""".strip(),
        metadata={"source": "kb:retrievers"},
    ),
    Document(
        page_content="""
Wikipedia is a useful free fallback when a curated knowledge base does not cover
a general topic. Internal documents should be preferred when they contain the
answer. Wikipedia results should be identified clearly and treated cautiously
for medical, legal, political, or otherwise sensitive claims.
""".strip(),
        metadata={"source": "kb:wikipedia_tip"},
    ),
    Document(
        page_content="""
When evidence is thin, ambiguous, or conflicting, an assistant must be transparent
about uncertainty. It should separate supported facts from inferences, avoid
inventing names or numbers, and suggest a more precise follow-up query when the
available sources are insufficient.
""".strip(),
        metadata={"source": "kb:honesty"},
    ),
    Document(
        page_content="""
Grounded answers connect each important claim to retrieved evidence. Citations
should appear near the supported claim, use stable source identifiers, and never
be fabricated. A concise answer should lead with the conclusion and cite only
the sources that materially support it.
""".strip(),
        metadata={"source": "kb:grounding"},
    ),
    Document(
        page_content="""
For straightforward questions, answers should be concise and well structured.
Start with the direct answer, add a short explanation, and include inline source
citations. Longer explanations are appropriate when the user explicitly asks for
a tutorial or when important safety caveats are required.
""".strip(),
        metadata={"source": "kb:style"},
    ),
]

# FakeEmbeddings is intentionally lightweight and requires no API key.
# It is suitable for this exercise, although it does not provide meaningful
# semantic embeddings like a trained sentence-transformer would.
embeddings = FakeEmbeddings(size=256)

# Build the in-memory FAISS vector store and expose a top-3 retriever.
vector_store = FAISS.from_documents(kb_docs, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print("KB ready with", len(kb_docs), "documents")
print("Retriever top-k:", retriever.search_kwargs["k"])


## 2) Open source external tool: Wikipedia search

In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper

wiki = WikipediaAPIWrapper(
    lang="en",
    top_k_results=3,
    doc_content_chars_max=1200,
)

def _wiki_slug(title: str) -> str:
    """Create the identifier used in citations."""
    return "_".join(title.strip().split())

def wiki_search(query: str, k: int = 2):
    """
    Search Wikipedia and return at most k short, structured snippets.

    Returns:
        (snippets, error)
        snippets: [{"title", "display_title", "summary", "url"}, ...]
        error: None on success, otherwise a readable error message.
    """
    clean_query = query.strip()
    if not clean_query:
        return [], "The Wikipedia query is empty."

    try:
        documents = wiki.load(clean_query)[:k]
        snippets = []

        for document in documents:
            metadata = document.metadata or {}
            display_title = (
                metadata.get("title")
                or metadata.get("page")
                or "Wikipedia"
            )
            summary = " ".join(document.page_content.split())

            if not summary:
                continue

            snippets.append(
                {
                    "title": _wiki_slug(display_title),
                    "display_title": display_title,
                    "summary": summary[:700],
                    "url": metadata.get("source"),
                }
            )

        if not snippets:
            return [], f"No Wikipedia result was found for: {clean_query}"

        return snippets, None

    except Exception as exc:
        return [], f"Wikipedia search failed: {type(exc).__name__}: {exc}"

preview, preview_error = wiki_search("Python programming", k=1)
print("Preview:", preview)
print("Error:", preview_error)


## 3) Simple planner (rule-based)

In [ ]:
import re

kb_keywords = {
    "agentic",
    "agent",
    "agents",
    "tool",
    "tools",
    "retriever",
    "retrieval",
    "rag",
    "citation",
    "citations",
    "cite",
    "ground",
    "grounded",
    "grounding",
    "evidence",
    "uncertain",
    "uncertainty",
    "honest",
    "honesty",
    "transparent",
    "transparency",
    "wikipedia",
    "answer style",
}

def _normalize_words(text: str) -> set[str]:
    return set(re.findall(r"[a-zA-Z]+", text.lower()))

def plan(question: str):
    """
    Prefer the internal KB when the question contains a known KB topic.
    Otherwise, use Wikipedia as the external fallback.
    """
    normalized_question = question.lower()
    words = _normalize_words(question)

    matched = sorted(
        keyword
        for keyword in kb_keywords
        if (
            keyword in words
            or (" " in keyword and keyword in normalized_question)
        )
    )

    if matched:
        return {
            "action": "kb",
            "reason": "The question matches a topic covered by the local KB.",
            "matched_keywords": matched,
        }

    return {
        "action": "wiki",
        "reason": "No known KB topic matched, so Wikipedia is used as fallback.",
        "matched_keywords": [],
    }

print(plan("How should an agent ground answers with citations?"))
print(plan("Who created Python?"))


## 4) Answer function with stub or tiny HF model

In [ ]:
from functools import lru_cache
import re

from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models.fake import FakeListChatModel
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

prompt = ChatPromptTemplate.from_template(
    """You are a helpful agentic assistant.

Use only the evidence provided below.
If the evidence is weak or missing, say so and suggest a follow-up query.
Cite sources inline using [kb:source] or [wiki:Title].

Question:
{question}

Knowledge-base context:
{context}

Wikipedia snippets:
{wiki}

Answer:
"""
)

def _split_sentences(text: str) -> list[str]:
    """A lightweight sentence splitter sufficient for short snippets."""
    return [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+", text.strip())
        if sentence.strip()
    ]

def _lexical_score(question: str, text: str) -> int:
    question_words = _normalize_words(question)
    text_words = _normalize_words(text)
    return len(question_words & text_words)

def _best_kb_document(question: str, documents):
    """Select the most lexically relevant document among retrieved results."""
    if not documents:
        return None
    return max(
        documents,
        key=lambda document: _lexical_score(question, document.page_content),
    )

def _draft_grounded_answer(question, documents, wiki_snippets, wiki_error):
    """
    Build a deterministic evidence-based draft.
    FakeListChatModel will return this draft as the default stub response.
    """
    if documents:
        best_document = _best_kb_document(question, documents)
        source = best_document.metadata.get("source", "kb:unknown")
        sentences = _split_sentences(best_document.page_content)
        evidence = " ".join(sentences[:2])

        if _lexical_score(question, best_document.page_content) == 0:
            return (
                "The retrieved KB evidence appears too weak to answer this "
                f"question confidently. The closest passage says: {evidence} "
                f"[{source}] Try a more specific query or use an external source."
            )

        return (
            f"{evidence} [{source}] "
            "This answer is limited to the evidence available in the local KB."
        )

    if wiki_snippets:
        snippet = wiki_snippets[0]
        sentences = _split_sentences(snippet["summary"])
        evidence = " ".join(sentences[:2])
        return (
            f"{evidence} [wiki:{snippet['title']}] "
            "This answer is based on the retrieved Wikipedia summary."
        )

    error_detail = f" ({wiki_error})" if wiki_error else ""
    return (
        "I do not have enough evidence to answer this question reliably"
        f"{error_detail}. Try a more precise query or add a relevant document "
        "to the knowledge base."
    )

@lru_cache(maxsize=1)
def get_tiny_generator(model_id: str = "sshleifer/tiny-gpt2"):
    """
    Load an optional tiny local Hugging Face model.
    The default exercise path does not load it.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    return pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        torch_dtype=torch.float32,
        device=-1,
    )

def summarize_with_tiny(prompt_text: str, max_new_tokens: int = 120):
    generator = get_tiny_generator()
    output = generator(
        prompt_text,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    return output[0]["generated_text"].strip()

def answer_question(question: str, use_tiny_model: bool = False):
    """
    Plan, retrieve evidence, generate an answer, and expose all used sources.

    The default uses FakeListChatModel, so no API key or model download is needed.
    Set use_tiny_model=True only to try the optional local Hugging Face model.
    """
    selected_plan = plan(question)

    documents = []
    wiki_snippets = []
    wiki_error = None

    if selected_plan["action"] == "kb":
        documents = retriever.invoke(question)
    else:
        wiki_snippets, wiki_error = wiki_search(question)

    context_text = (
        "\n".join(
            f"[{document.metadata.get('source', 'kb:unknown')}] "
            f"{document.page_content}"
            for document in documents
        )
        or "No KB context."
    )

    wiki_text = (
        "\n".join(
            f"[wiki:{snippet['title']}] {snippet['summary']}"
            for snippet in wiki_snippets
        )
        or "No Wikipedia snippets."
    )

    messages = prompt.format_messages(
        question=question,
        context=context_text,
        wiki=wiki_text,
    )

    grounded_draft = _draft_grounded_answer(
        question,
        documents,
        wiki_snippets,
        wiki_error,
    )

    if use_tiny_model:
        final_answer = summarize_with_tiny(messages[0].content)

        # Tiny language models may omit citations, so preserve traceability.
        expected_sources = [
            document.metadata.get("source")
            for document in documents
            if document.metadata.get("source")
        ] + [
            f"wiki:{snippet['title']}"
            for snippet in wiki_snippets
        ]

        if expected_sources and not re.search(r"\[(kb|wiki):", final_answer):
            final_answer += "\n\nSources: " + " ".join(
                f"[{source}]" for source in expected_sources[:3]
            )

        if not final_answer:
            final_answer = grounded_draft
    else:
        stub = FakeListChatModel(responses=[grounded_draft])
        final_answer = stub.invoke(messages).content

    return {
        "plan": selected_plan,
        "kb_sources": [
            document.metadata.get("source")
            for document in documents
        ],
        "wiki_sources": [
            snippet.get("title")
            for snippet in wiki_snippets
        ],
        "wiki_error": wiki_error,
        "answer": final_answer,
    }


## 5) Quick check on sample questions

In [ ]:
tests = [
    # Covered by the local KB.
    "How should an agent ground its answers and cite evidence?",

    # Not covered by the local KB, so the planner should use Wikipedia.
    "Who created the Python programming language?",

    # Ambiguous/general question not represented by the KB.
    "What is the history of the word intelligence?",
]

for number, question in enumerate(tests, start=1):
    result = answer_question(question, use_tiny_model=False)

    print("=" * 90)
    print(f"TEST {number}")
    print("Question:", question)
    print("Plan:", result["plan"])
    print("KB sources:", result["kb_sources"] or "None")
    print("Wikipedia sources:", result["wiki_sources"] or "None")

    if result["wiki_error"]:
        print("Wikipedia warning:", result["wiki_error"])

    print("Final answer:", result["answer"])
    print()

# Basic checks required by the exercise.
kb_result = answer_question(tests[0], use_tiny_model=False)
external_result = answer_question(tests[1], use_tiny_model=False)

assert kb_result["plan"]["action"] == "kb"
assert kb_result["kb_sources"]
assert external_result["plan"]["action"] == "wiki"

print("Quick checks completed.")


## Submission notes

- Run all cells from top to bottom in Google Colab.
- The default path uses `FakeListChatModel`; no API key is required.
- `use_tiny_model=True` is optional and downloads a Hugging Face model.
- Wikipedia needs internet access and may occasionally return no result; the answer function handles this gracefully.
- Save the completed notebook and push it to the requested GitHub repository.
